Web Crawler data flow: 

1. Take seed URL from frontier and request IP from DNS
2. Fetch HTML from external server using IP
3. Extract text data from the HTML.
4. Store the text data in a database.
5. Extract any linked URLs from the web pages and add them to the list (Frontier Queue) of URLs to crawl.
6. Repeat steps 1-5 until all URLs have been crawled.

In [19]:
# TESTING_SEED_URL = "https://softwarica.edu.np/courses"
# TESTING_ALLOWED_DOMAIN = "softwarica.edu.np"

# TESTING_CRAWL_DELAY_SECONDS = 5*24*60*60      # 5 days
# TESTING_MAX_PAGES = 100               
# TESTING_USER_AGENT = "SoftwaricaVerticalSearchBot/1.0 (+educational IR project)"

In [20]:
base_url = "https://pureportal.coventry.ac.uk/en/organisations/centre-for-healthcare-and-community-transformation/"

In [21]:
SEED_URL = "https://pureportal.coventry.ac.uk/en/organisations/centre-for-healthcare-and-community-transformation/publications/"
ALLOWED_DOMAIN = "pureportal.coventry.ac.uk"
CRAWL_DELAY_SECONDS = 24*60*60*30*3    # 3 months      
MAX_PAGES = 100               
USER_AGENT = "CoventryVerticalSearchBot/1.0 (+educational IR project)"
my_current_chrome_version = 146


Ensuring Politness steps: 

1. To make this clear, the steps would be:
2. Fetch the `robots.txt` file for the domain.
3. Parse the `robots.txt` file and store it in the database (MongoDB).
4. When we pull a URL off the queue, check the rules stored in the database (MongoDB) for that domain.
5. If the URL is disallowed, ack the message and move on to the next URL.
6. If the URL is allowed, check the `Crawl-delay` directive.

IF Crawler is failed

7. If the Crawl-delay time has not passed since the last crawl, use ChangeMessageVisibility to extend the visibility timeout and defer reprocessing.
8. If the Crawl-delay time has passed, crawl the page and update the last crawl time for the domain.

In [22]:
from urllib.parse import urljoin, urlparse
from urllib.robotparser import RobotFileParser
import requests
import json
from bs4 import BeautifulSoup, NavigableString
import re

import undetected_chromedriver as uc
import time

In [23]:
# fetching the content of the robots.txt file 

def fetch_robots(base_url, USER_AGENT):
    parsed_url = urlparse(base_url)     # Example: ParseResult(scheme='https', netloc='softwarica.edu.np', path='/courses', params='', query='', fragment='')
    robots_url = f"{parsed_url.scheme}://{parsed_url.netloc}/robots.txt"
    rfp = RobotFileParser()
    rfp.set_url(robots_url)

    try: 
        res = requests.get(robots_url, headers={"User-Agent": USER_AGENT}, timeout=10)
        if res.status_code == 200: 
            rfp.parse(res.text.splitlines())
        else: 
            rfp = None
    except BaseException as err: 
        print(f"Error: {err}")
        rfp = None
        
    return rfp

In [24]:
def can_fetch(rfp, url, USER_AGENT=USER_AGENT):
    if rfp is None: 
        return True
    return rfp.can_fetch(USER_AGENT, url)

In [25]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as ec

In [26]:
# setting headless driver of chrome browser for crawling

def setup_headless_driver():
    options = Options()
    options.add_argument("--headless=new") 
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument(f"user-agent={USER_AGENT}")
    
    driver = webdriver.Chrome(options=options)
    return driver

In [27]:

# 
def setup_driver(current_chrome_version):
    # getting chrome options 
    options = uc.ChromeOptions()
    driver = uc.Chrome(options=options, version_main=current_chrome_version)
    return driver
    


In [28]:
# bypassing the Cloudflare bot by using undetected-chromedriver
driver = setup_driver(my_current_chrome_version)

driver.get(SEED_URL)

time.sleep(10)

html_content = driver.page_source
soup = BeautifulSoup(html_content, "html.parser")

print("Page loaded. Extracting...\n")

Page loaded. Extracting...



In [29]:
from collections import deque, Counter
import time

In [30]:

# defining a crawler

def crawl(seed_url, max_pages, user_agent, allowed_domain, crawl_delay_seconds):
    """crawls through the entire links of the seed url and returns the number of crawled links

    Args:
        seed_url (str): seed url 
        max_pages (int): up to max pages for crawling
        user_agent (str): name of user agent
        allowed_domain (str): allowed domain to crawl
        crawl_delay_seconds (time): time (seconds) in integer for ensuring politness

    Returns:
        int: number of crawled links
    """
    
    rfp = fetch_robots(seed_url, user_agent)
    
    # storing all extracted URLs to visit
    frontier_queue = deque([seed_url])
    
    visited_links = set()  # not visited any link so empty
    
    crawled_count = 0  # not crawled yet so empty
    
    print(f"{10*"="}Initializing browser and bypassing Cloudflare {10*"="}")
    driver = setup_driver(my_current_chrome_version)
    
    try: 
        # loop until all links are visited
        while frontier_queue and crawled_count < max_pages:
            
            url = frontier_queue.popleft()
            print('URL', url)
            
            if url in visited_links:
                continue
            
            if not can_fetch(rfp, url):
                print(f"Blocked by robots.txt: {url}")
                continue
            
            try:
                driver.get(url)
                WebDriverWait(driver, 10).until(
                    ec.presence_of_element_located((By.TAG_NAME, "body"))
                )
                
                # wait for 5 sec between request (ensuring politness --> DO NOT hit the host server)
                time.sleep(5) 
                html = driver.page_source
                
            except BaseException as err:
                    print(f"Error while fetching {url} url: {err}")
                    continue
            title, text, links = extract_content(url, allowed_domain)
            
            # TODO: store raw pages in DB
            
            
            
            crawled_count += 1
            
            print(f"{crawled_count} Crawled: {url}")
            
            for link in links:
                # if link is not visited then add it to the url frontier queue
                if link not in visited_links:
                    frontier_queue.append(link)
            
                # wait for 5 sec between request (ensuring politness --> DO NOT hit the host server)
            
            time.sleep(crawl_delay_seconds)
            
    except BaseException as err:
        print(f"Error while crawling: {err}")
        
    finally:
        driver.quit()
        print(f"{10*"="} Closed Headless Chrome driver {10*"="}")
        
    # TODO: store crawl logs in DB
    print(f"Crawling completed. {crawled_count} pages crawled and needs to be stored in DB")
    return crawled_count
    

In [31]:

# crawl coventry pure potal
# crawl(
#     seed_url=MAIN_SEED_URL,
#     max_pages=MAIN_MAX_PAGES,
#     user_agent=MAIN_USER_AGENT,
#     allowed_domain=MAIN_ALLOWED_DOMAIN,
#     crawl_delay_seconds=10
# )

Note: Rate limiting (avoiding system crash by requesting to crawl) is important. Sliding window algorithm can be used to track the number of requests per domain per second

In [36]:
def extract_research_output(base_url, current_chrome_version):
    driver = setup_driver(current_chrome_version)

    # for all research output data
    extracted_data = []

    # tracking page number since, research ouptut is divided into page 1, 2
    page_number = 0

    while True:
        # making paginated url
        current_url = f"{base_url}publications/?page={page_number}"
        print(f"Fetching: {current_url}")
        
        driver.get(current_url)
        
        # waiting for cloudflare on the first page, subsequent pages might load faster
        if page_number == 0:
            time.sleep(10)
        else:
            # waiting shorter for subsequent pages assuming cloudflare is already passed
            time.sleep(5)
        
        # extracting all HTML content of the page
        html_content = driver.page_source
        soup = BeautifulSoup(html_content, "html.parser")
        
        
        # finding all research outputs on the current page
        results = soup.find_all('div', class_='rendering_researchoutput')
        
        # if no results are found on this page, we have reached the end
        if not results:
            print(f"No more results found. Exiting loop.")
            break
            
        print(f"Found {len(results)} outputs on page {page_number}. Extracting...\n")
        
        for div in results:
            title = None
            title_link = None
            
            # extracting title and its link
            h3 = div.find('h3', class_='title')
            if h3:
                a_tag = h3.find('a', class_='link')
                if a_tag:
                    title = a_tag.text.strip()
                    title_link = a_tag.get('href')
                else:
                    # fallback in case there is no link
                    title = h3.text.strip()
                    
            # extracting authors along with link
            authors = []
            if h3:
                # authors are floating between the h3 tag and the date span
                for sibling in h3.next_siblings:
                    # stop looking for authors once we hit the date span
                    if sibling.name == 'span' and sibling.get('class') and 'date' in sibling.get('class'):
                        break
                        
                    # if the sibling is raw text
                    if isinstance(sibling, NavigableString):
                        # cleaning up the raw text to remove dangling commas and whitespace
                        text = sibling.strip(', & \n\r\t')
                        if text:
                            authors.append({'name': text, 'link': None})
                            
                    # if the sibling is an a tag
                    elif sibling.name == 'a' and sibling.get('class') and 'person' in sibling.get('class'):
                        authors.append({
                            'name': sibling.text.strip(),
                            'link': sibling.get('href')
                        })

            # extracting publish date
            date_span = div.find('span', class_='date')
            publish_date = date_span.text.strip() if date_span else None
            
            # extracting journal name
            journal_span = div.find('span', class_='journal')
            journal_name = journal_span.text.strip() if journal_span else None
            
            # extracting journal volume
            volume_span = div.find('span', class_='volume')
            journal_volume = volume_span.text.strip() if volume_span else None
            
            # extracting number of pages
            pages_span = div.find('span', class_='numberofpages')
            number_of_pages = pages_span.text.strip() if pages_span else None

            research_output = {
                'title': title,
                'title_link': title_link,
                'authors': authors,
                'publish_date': publish_date,
                'journal_name': journal_name,
                'journal_volume': journal_volume,
                'number_of_pages': number_of_pages
            }
            
            if research_output['title'] is None:
                continue
            
            extracted_data.append(research_output)

        # adding page number to go to the next page
        page_number += 1

    driver.quit()
    print("Driver is closed...")

    # saving json file
    output_filename = "all_research_outputs.json"
    # with open(f"../data/{output_filename}", 'w', encoding='utf-8') as json_file:
    #     json.dump(extracted_data, json_file, indent=4, ensure_ascii=False)

    print(f"\nSuccessfully saved a total of {len(extracted_data)} research outputs to '{output_filename}'!")
    
    return extracted_data

In [37]:
extract_research_output(base_url, my_current_chrome_version)

Fetching: https://pureportal.coventry.ac.uk/en/organisations/centre-for-healthcare-and-community-transformation/publications/?page=0
Found 96 outputs on page 0. Extracting...

Fetching: https://pureportal.coventry.ac.uk/en/organisations/centre-for-healthcare-and-community-transformation/publications/?page=1
Found 45 outputs on page 1. Extracting...

Fetching: https://pureportal.coventry.ac.uk/en/organisations/centre-for-healthcare-and-community-transformation/publications/?page=2
No more results found. Exiting loop.
Driver is closed...

Successfully saved a total of 75 research outputs to 'all_research_outputs.json'!


[{'title': "A Cross-Sectional Study of Postgraduate Students' Mental Well-Being: Exploring the Relationship Between Mental Well-Being, Perceived Stress, Academic Self-Efficacy, and Self-Efficacy for Self-Regulated Learning",
  'title_link': 'https://pureportal.coventry.ac.uk/en/publications/a-cross-sectional-study-of-postgraduate-students-mental-well-bein/',
  'authors': [{'name': 'Bisal, N.', 'link': None},
   {'name': 'Brookes-Smith, C.',
    'link': 'https://pureportal.coventry.ac.uk/en/persons/celine-brookes-smith/'},
   {'name': 'Patel, R., Sharp, S.', 'link': None},
   {'name': 'Lycett, D.',
    'link': 'https://pureportal.coventry.ac.uk/en/persons/deborah-lycett/'},
   {'name': 'Turner, A.',
    'link': 'https://pureportal.coventry.ac.uk/en/persons/andy-turner/'},
   {'name': 'Whelan, M.',
    'link': 'https://pureportal.coventry.ac.uk/en/persons/maxine-whelan/'}],
  'publish_date': 'May 2026',
  'journal_name': 'In: Health Science Reports.',
  'journal_volume': '9',
  'number_o

In [38]:


# storing all data across all pages here
extracted_profiles = []

# starting pagination at page 0
page_number = 0

while True:
    # constructing the paginated url
    current_url = f"{base_url}persons/?page={page_number}"
    print(f"Fetching: {current_url}")
    
    driver.get(current_url)
    
    
    # waiting for cloudflare on the first page, subsequent pages might load faster
    if page_number == 0:
        time.sleep(10)
    else:
        # waiting a shorter time for subsequent pages assuming cloudflare is already passed
        time.sleep(5)
        
    html_content = driver.page_source
    soup = BeautifulSoup(html_content, "html.parser")
    
    # targeting the main container for each profile card
    results = soup.find_all('div', class_='result-container')
    
    # breaking the loop if no results are found on this page
    if not results:
        print("No more profiles found. Exiting loop.")
        break
        
    print(f"Found {len(results)} profiles on page {page_number}. Extracting...\n")
    
    if page_number == 3:
        break
    
    for div in results:
        # skipping if this container doesn't actually hold a person profile
        if not div.find('div', class_='rendering_person'):
            continue
            
        # extracting image url
        image_url = None
        img_tag = div.find('img', class_='image')
        if img_tag and img_tag.get('src'):
            image_url = img_tag.get('src')
            # appending the base domain because pureportal often uses relative image paths
            if image_url.startswith('/'):
                image_url = f"https://pureportal.coventry.ac.uk{image_url}"

        # extracting name and profile link
        name = None
        profile_link = None
        h3 = div.find('h3', class_='title')
        if h3:
            a_tag = h3.find('a')
            if a_tag:
                name = a_tag.text.strip()
                profile_link = a_tag.get('href')
            else:
                name = h3.text.strip()

        # extracting relations / organisations
        organizations = []
        org_ul = div.find('ul', class_='relations organisations')
        if org_ul:
            # finding all list items within the relations ul
            for li in org_ul.find_all('li'):
                org_text = li.text.strip()
                if org_text:
                    organizations.append(org_text)

        # extracting person type (e.g., academic staff)
        type_p = div.find('p', class_='type')
        person_type = type_p.text.strip() if type_p else None

        # extracting active publication years (from the stacked-trend-widget)
        start_year = None
        end_year = None
        years = div.find_all('span', class_='stacked-trend-graph-year')
        
        if years:
            # assigning the first span as the start year
            start_year = years[0].text.strip()
            # assigning the last span as the end year (if there is more than one year)
            if len(years) > 1:
                end_year = years[-1].text.strip()
            else:
                # setting the end year same as start year if only one year is listed
                end_year = start_year 

        # compiling into a dictionary
        profile_data = {
            'name': name,
            'profile_link': profile_link,
            'image_url': image_url,
            'organizations': organizations,
            'person_type': person_type,
            'active_years': {
                'start': start_year,
                'end': end_year
            }
        }
        print(profile_data)
        
        extracted_profiles.append(profile_data)

    # incrementing page number to go to the next page
    page_number += 1

# closing the browser
# driver.quit()

# saving json file
output_filename = "all_profiles.json"
with open(f"../data/{output_filename}", 'w', encoding='utf-8') as json_file:
    json.dump(extracted_profiles, json_file, indent=4, ensure_ascii=False)

print(f"\nSuccessfully saved a total of {len(extracted_profiles)} profiles to '{output_filename}'!")

Fetching: https://pureportal.coventry.ac.uk/en/organisations/centre-for-healthcare-and-community-transformation/persons/?page=0
Found 50 profiles on page 0. Extracting...

{'name': 'Sally Abbott', 'profile_link': 'https://pureportal.coventry.ac.uk/en/persons/sally-abbott/', 'image_url': 'https://pureportal.coventry.ac.uk/files-asset/56406205/Photo.jpeg?w=50&f=jpg', 'organizations': ['Centre for Healthcare and Community Transformation - Assistant Professor'], 'person_type': 'Person: Teaching and Research', 'active_years': {'start': '2016', 'end': '2025'}}
{'name': 'Abidemi Funmi Adegaye', 'profile_link': 'https://pureportal.coventry.ac.uk/en/persons/abidemi-funmi-adegaye/', 'image_url': 'https://pureportal.coventry.ac.uk/assets/no-portrait-473c6d005990baa1f418d9c668dcd4ec.png', 'organizations': ['Centre for Healthcare and Community Transformation'], 'person_type': 'Person: Masters Student', 'active_years': {'start': None, 'end': None}}
{'name': 'Nazia Afreen', 'profile_link': 'https://p